In [ ]:
import os
from google.colab import drive


print("Cloning repository and installing dependencies...")
!git clone https://github.com/duyngtr16061999/AI6130_Assignment2
!pip install fire datasets
!pip install transformers==4.38.0 accelerate==0.28.0
!pip uninstall peft -y

# Ensure we are in the correct directory and WandB is disabled
os.chdir('/kaggle/working/AI6130_Assignment2')
os.environ["WANDB_MODE"] = "disabled"

# Turn ON High-speed inference (KV Cache) for evaluation
!sed -i 's/use_cache=False/use_cache=True/g' evaluate.py
print("🚀 High-speed inference (KV Cache) enabled!")

# Bonus Experiment Parameters
model = "openlm-research/open_llama_3b_v2"
model_name = "open_llama_3b_v2"
adapter_lower, adapter_eval = "lora", "LoRA"
# Save directly to Google Drive
out_dir = f"/content/drive/MyDrive/AI6130_Assignment2_Results/{model_name}-bonus-10k-3ep"
bonus_datasets = [ "AQuA" ,"AddSub","MultiArith", "SingleEq", "gsm8k", "SVAMP"]

# Ensure output directory exists on Drive
os.makedirs(out_dir, exist_ok=True)

print("\n🌟 STARTING BONUS EXPERIMENT...")
print(f"{'='*60}")
print(f"🎯 TRAINING: {model_name} | 3 Epochs | 10k Dataset")
print(f"{'='*60}")

# 1. Finetune (3 Epochs + 10k Dataset) - Forced Single GPU
train_cmd = (
    f"CUDA_VISIBLE_DEVICES=0 python finetune.py --base_model '{model}' "
    f"--data_path './ft-training_set/math_10k.json' "
    f"--output_dir '{out_dir}' --batch_size 4 --micro_batch_size 4 "
    f"--num_epochs 3 --learning_rate 3e-4 --cutoff_len 256 "
    f"--val_set_size 120 --adapter_name {adapter_lower}"
)
!{train_cmd}

# 2. Evaluate on ALL datasets - Forced Single GPU
for dataset in bonus_datasets:
    print(f"\n---> 🧪 EVALUATING BONUS: {dataset}")
    eval_cmd = (
        f"CUDA_VISIBLE_DEVICES=0 python evaluate.py --adapter {adapter_eval} "
        f"--dataset {dataset} --base_model '{model}' "
        f"--lora_weights '{out_dir}'"
    )
    !{eval_cmd}

print("\n✅ BONUS EXPERIMENT COMPLETE!")

Cloning repository and installing dependencies...
Cloning into 'AI6130_Assignment2'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 126 (delta 3), reused 1 (delta 0), pack-reused 117 (from 2)
Receiving objects: 100% (126/126), 72.34 MiB | 18.85 MiB/s, done.
Resolving deltas: 100% (21/21), done.
Updating files: 100% (88/88), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.